# Fetch Corners Data from StatsBomb

Pulls corner-kick events from the StatsBomb API for the configured competitions/seasons,
links each corner delivery to any shot that followed in the same possession,
derives `SP_outcome` and all columns expected by the app, and saves one
parquet file per competition to `Data/Corners/`.

**Naming convention:** `{Competition Name} - Corners {Season Label}.parquet`  
e.g. `Premier League - Corners 2025-2026.parquet`

**Run locally** — StatsBomb API is IP-restricted.

In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────────
# pip install statsbombpy pandas pyarrow fastparquet

import re
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import numpy as np
import pandas as pd
from statsbombpy import sb

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────

ACCOUNTS = [
    {"user": "m.pulley@az.nl",         "passwd": "SwHrVcks"},
    {"user": "JACK71299@HOTMAIL.CO.UK", "passwd": "J7rB7aP2"},
]

# Season IDs to fetch.  316 = 2026, 318 = 2025-2026
TARGET_SEASON_IDS = [316, 318]

# Label appended to the filename for each season_id
SEASON_LABELS = {
    316: "2026",
    318: "2025-2026",
}

OUTPUT_DIR  = Path("../Data/Corners")
MAX_WORKERS = 6          # parallel match fetches per competition
SKIP_EXISTING = True     # set False to re-fetch and overwrite

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# ── Helpers ───────────────────────────────────────────────────────────────────

def log(msg: str) -> None:
    print(msg, flush=True)


def try_accounts(func, *args, **kwargs):
    """Try each credentials pair; return first non-empty result."""
    for idx, creds in enumerate(ACCOUNTS, 1):
        try:
            result = func(*args, creds=creds, **kwargs)
            if isinstance(result, pd.DataFrame) and not result.empty:
                return result
            if isinstance(result, (list, dict)) and result:
                return result
            if result is not None:
                return result
        except Exception as e:
            log(f"  Account {idx} failed ({func.__name__}): {e}")
    return None


def safe_filename(name: str) -> str:
    return re.sub(r'[<>:"/\\|?*\t]', '', name).strip()


def ts_to_seconds(ts) -> float:
    """Convert StatsBomb HH:MM:SS.mmm timestamp to total seconds."""
    if pd.isna(ts):
        return np.nan
    parts = str(ts).split(":")
    return int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2])

In [ ]:
# ── SP_outcome derivation ─────────────────────────────────────────────────────
#
# Rules (derived from existing parquet analysis):
#
#  pass.outcome.name  |  shot present?  |  timing          → SP_outcome
#  ───────────────────┼─────────────────┼──────────────────────────────────────
#  Out                |  —              |  —               → Wayward
#  Unknown            |  —              |  —               → 50/50
#  Incomplete         |  —              |  —               → First contact lost
#  NaN (complete)     |  Yes            |  dt ≤ 3 s        → First contact - shot within 3 seconds
#  NaN (complete)     |  Yes            |  dt > 3 s        → No first contact - shot
#  NaN (complete)     |  No             |  pass_end in box → Short in area
#  NaN (complete)     |  No             |  otherwise       → No first contact - no shot
#
# "Short in area" = completed delivery whose end-point is inside the penalty area
# (pass_end_location_x > 102 on StatsBomb 120×80 pitch) AND close to goal-mouth.
# In practice all Short-in-area rows had pass_end_x > 102 so we use that threshold.

PENALTY_BOX_X = 102.0   # StatsBomb x; box starts here
FIRST_CONTACT_THRESHOLD = 3.0   # seconds


def derive_sp_outcome(row) -> str:
    outcome = row.get("pass.outcome.name")
    has_shot = pd.notna(row.get("shot.outcome.name"))

    if outcome == "Out":
        return "Wayward"
    if outcome == "Unknown":
        return "50/50"
    if outcome == "Incomplete":
        return "First contact lost"

    # Completed delivery (outcome is NaN)
    if has_shot:
        dt = row.get("_dt", np.nan)
        if pd.notna(dt) and dt <= FIRST_CONTACT_THRESHOLD:
            return "First contact - shot within 3 seconds"
        return "No first contact - shot"

    end_x = row.get("pass_end_location_x", np.nan)
    if pd.notna(end_x) and end_x > PENALTY_BOX_X:
        return "Short in area"
    return "No first contact - no shot"

In [ ]:
# ── Per-match extraction ──────────────────────────────────────────────────────

def extract_corners_from_match(match_id: int, match_label: str) -> pd.DataFrame | None:
    """
    Fetch all events for one match, extract corner deliveries, link to shots
    in the same possession, derive SP_outcome, and return a tidy DataFrame.
    Returns None on failure.
    """
    raw = try_accounts(sb.events, match_id=int(match_id))
    if raw is None or (isinstance(raw, pd.DataFrame) and raw.empty):
        return None
    if not isinstance(raw, pd.DataFrame):
        try:
            raw = pd.DataFrame(raw)
        except Exception:
            return None

    # ── 1. Identify corner pass events ────────────────────────────────────────
    passes = raw[raw["type"].apply(lambda x: x.get("name") if isinstance(x, dict) else x) == "Pass"].copy()

    def get_nested(series, *keys):
        """Extract a nested dict key from a Series of dicts."""
        result = series
        for key in keys:
            result = result.apply(lambda x: x.get(key) if isinstance(x, dict) else np.nan)
        return result

    if "pass" not in passes.columns:
        return None

    pass_type = get_nested(passes["pass"], "type", "name")
    corners = passes[pass_type == "Corner"].copy()
    if corners.empty:
        return None

    # ── 2. Flatten pass fields ────────────────────────────────────────────────
    corners["pass_timestamp"]       = corners["timestamp"]
    corners["pass_team_name"]       = corners["team"].apply(lambda x: x.get("name") if isinstance(x, dict) else x)
    corners["Taker"]                = corners["player"].apply(lambda x: x.get("name") if isinstance(x, dict) else x)
    corners["pass_position"]        = corners["position"].apply(lambda x: x.get("name") if isinstance(x, dict) else np.nan)
    corners["pass.height.name"]     = get_nested(corners["pass"], "height", "name")
    corners["pass.body_part.name"]  = get_nested(corners["pass"], "body_part", "name")
    corners["pass.outcome.name"]    = get_nested(corners["pass"], "outcome", "name")
    corners["pass.technique.name"]  = get_nested(corners["pass"], "technique", "name")

    corners["pass_location_x"]      = corners["location"].apply(lambda x: x[0] if isinstance(x, list) and len(x) >= 2 else np.nan)
    corners["pass_location_y"]      = corners["location"].apply(lambda x: x[1] if isinstance(x, list) and len(x) >= 2 else np.nan)

    end_loc = corners["pass"].apply(lambda x: x.get("end_location") if isinstance(x, dict) else None)
    corners["pass_end_location_x"]  = end_loc.apply(lambda x: x[0] if isinstance(x, list) and len(x) >= 2 else np.nan)
    corners["pass_end_location_y"]  = end_loc.apply(lambda x: x[1] if isinstance(x, list) and len(x) >= 2 else np.nan)

    corners["Minute"]  = corners["minute"]
    corners["Second"]  = corners["second"]
    corners["match_id"] = match_id
    corners["Match"]   = match_label

    # ── 3. Find the first shot in the same possession ─────────────────────────
    shots = raw[raw["type"].apply(lambda x: x.get("name") if isinstance(x, dict) else x) == "Shot"].copy()

    if not shots.empty and "shot" in shots.columns:
        shots["shot_team_name"]      = shots["team"].apply(lambda x: x.get("name") if isinstance(x, dict) else x)
        shots["Shooter"]             = shots["player"].apply(lambda x: x.get("name") if isinstance(x, dict) else x)
        shots["shot_position"]       = shots["position"].apply(lambda x: x.get("name") if isinstance(x, dict) else np.nan)
        shots["shot.body_part.name"] = get_nested(shots["shot"], "body_part", "name")
        shots["shot.outcome.name"]   = get_nested(shots["shot"], "outcome", "name")
        shots["shot.statsbomb_xg"]   = shots["shot"].apply(lambda x: x.get("statsbomb_xg") if isinstance(x, dict) else np.nan)
        shots["shot_location_x"]     = shots["location"].apply(lambda x: x[0] if isinstance(x, list) and len(x) >= 1 else np.nan)
        shots["shot_location_y"]     = shots["location"].apply(lambda x: x[1] if isinstance(x, list) and len(x) >= 2 else np.nan)
        shots["shot_location_z"]     = shots["location"].apply(lambda x: x[2] if isinstance(x, list) and len(x) >= 3 else np.nan)
        shots["shot_timestamp"]      = shots["timestamp"]

        shot_cols = ["possession", "shot_team_name", "Shooter", "shot_position",
                     "shot.body_part.name", "shot.outcome.name", "shot.statsbomb_xg",
                     "shot_location_x", "shot_location_y", "shot_location_z", "shot_timestamp"]
        shot_cols = [c for c in shot_cols if c in shots.columns]

        # Keep only the first shot per possession (highest index = first to appear)
        first_shots = shots[shot_cols].drop_duplicates(subset=["possession"], keep="first")

        corners = corners.merge(first_shots, on="possession", how="left")
    else:
        for col in ["shot_team_name", "Shooter", "shot_position", "shot.body_part.name",
                    "shot.outcome.name", "shot.statsbomb_xg", "shot_location_x",
                    "shot_location_y", "shot_location_z", "shot_timestamp"]:
            corners[col] = np.nan

    # ── 4. Defensive setup (empty — not derivable from open-data events) ───────
    corners["Defensive_setup"] = ""

    # ── 5. SP_outcome ─────────────────────────────────────────────────────────
    corners["_pass_sec"] = corners["pass_timestamp"].apply(ts_to_seconds)
    corners["_shot_sec"] = corners["shot_timestamp"].apply(ts_to_seconds) if "shot_timestamp" in corners.columns else np.nan
    corners["_dt"] = corners["_shot_sec"] - corners["_pass_sec"]

    corners["SP_outcome"] = corners.apply(derive_sp_outcome, axis=1)

    # ── 6. Final column selection ─────────────────────────────────────────────
    output_cols = [
        "match_id", "Match", "possession",
        "pass_timestamp", "pass_team_name", "Taker", "pass_position",
        "pass.height.name", "pass.body_part.name", "pass.outcome.name", "pass.technique.name",
        "pass_location_x", "pass_location_y", "pass_end_location_x", "pass_end_location_y",
        "shot_timestamp", "shot_team_name", "Shooter", "shot_position",
        "shot.body_part.name", "shot.outcome.name", "shot.statsbomb_xg",
        "shot_location_x", "shot_location_y", "shot_location_z",
        "Defensive_setup", "Minute", "Second", "SP_outcome",
    ]
    for col in output_cols:
        if col not in corners.columns:
            corners[col] = np.nan

    return corners[output_cols].reset_index(drop=True)


print("extract_corners_from_match defined")

In [ ]:
# ── Per-competition pipeline ──────────────────────────────────────────────────

def process_competition(comp_id: int, season_id: int, comp_name: str) -> tuple[str, int]:
    season_label = SEASON_LABELS.get(season_id, str(season_id))
    out_path = OUTPUT_DIR / f"{safe_filename(comp_name)} - Corners {season_label}.parquet"

    if SKIP_EXISTING and out_path.exists():
        log(f"  [skip] {out_path.name}")
        return "skipped", 0

    # Fetch match list
    matches = try_accounts(sb.matches, competition_id=comp_id, season_id=season_id)
    if matches is None or (isinstance(matches, pd.DataFrame) and matches.empty):
        return "no_matches", 0

    matches["match_label"] = (
        matches["home_team"].apply(lambda x: x.get("home_team_name") if isinstance(x, dict) else str(x))
        + " - "
        + matches["away_team"].apply(lambda x: x.get("away_team_name") if isinstance(x, dict) else str(x))
    )
    match_ids   = matches["match_id"].dropna().astype(int).tolist()
    label_map   = dict(zip(matches["match_id"].astype(int), matches["match_label"]))

    if not match_ids:
        return "no_matches", 0

    log(f"  Fetching {len(match_ids)} matches ...")

    # Parallel event fetch
    frames = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(extract_corners_from_match, mid, label_map.get(mid, str(mid))): mid
                   for mid in match_ids}
        for fut in as_completed(futures):
            try:
                df = fut.result()
                if df is not None and not df.empty:
                    frames.append(df)
            except Exception as exc:
                log(f"    match {futures[fut]} error: {exc}")

    if not frames:
        return "no_corners", 0

    combined = pd.concat(frames, ignore_index=True)

    # Save
    try:
        combined.to_parquet(out_path, engine="fastparquet", index=False, compression="zstd")
    except Exception:
        combined.to_parquet(out_path, engine="pyarrow", index=False)

    log(f"  ✓ saved {len(combined):,} corners → {out_path.name}")
    return "ok", len(combined)


print("process_competition defined")

In [ ]:
# ── Main loop ─────────────────────────────────────────────────────────────────

# Load competition list
comps_path = Path("../Data/competitions.csv")
if comps_path.exists():
    all_comps = pd.read_csv(comps_path)
    log(f"Loaded {len(all_comps)} competitions from {comps_path}")
else:
    log("competitions.csv not found — fetching from StatsBomb ...")
    all_comps = try_accounts(sb.competitions)
    if all_comps is None:
        raise SystemExit("Could not load competitions — check credentials / connectivity.")

targets = (
    all_comps[all_comps["season_id"].isin(TARGET_SEASON_IDS)]
    .drop_duplicates(subset=["competition_id", "season_id"])
    .copy()
)
log(f"Competitions to process: {len(targets)} (seasons {TARGET_SEASON_IDS})\n")

ok_count = skipped = failed = 0

for _, row in targets.iterrows():
    comp_id   = int(row["competition_id"])
    season_id = int(row["season_id"])
    comp_name = str(row.get("competition_name", f"Competition_{comp_id}"))

    log(f"→ [{season_id}] {comp_name} (comp={comp_id})")
    status, n = process_competition(comp_id, season_id, comp_name)

    if status == "ok":       ok_count += 1
    elif status == "skipped": skipped  += 1
    else:
        log(f"  ✗ {status}")
        failed += 1

log(f"\n{'='*60}")
log(f"Done — saved: {ok_count}  |  skipped: {skipped}  |  failed/empty: {failed}")
log(f"\nFiles in {OUTPUT_DIR}:")
for f in sorted(OUTPUT_DIR.glob("*.parquet")):
    size_kb = f.stat().st_size // 1024
    log(f"  {f.name}  ({size_kb} KB)")

## Verify output
Spot-check one file to confirm columns and SP_outcome distribution match the expected format.

In [ ]:
# ── Verification ──────────────────────────────────────────────────────────────

files = sorted(OUTPUT_DIR.glob("*.parquet"))
if not files:
    print("No output files found.")
else:
    sample = pd.read_parquet(files[0])
    print(f"File: {files[0].name}")
    print(f"Shape: {sample.shape}")
    print(f"\nColumns ({len(sample.columns)}):")
    print(sample.columns.tolist())
    print(f"\nSP_outcome distribution:")
    print(sample["SP_outcome"].value_counts().to_string())
    print(f"\nTechnique distribution:")
    print(sample["pass.technique.name"].value_counts().to_string())
    print(f"\nShot rows: {sample['shot.outcome.name'].notna().sum()} / {len(sample)}")
    print(f"Goals:      {(sample['shot.outcome.name'] == 'Goal').sum()}")
    print(f"\nSample rows:")
    display(sample.head(3))